# QuPath GeoJSON to Leica LMD XML
Converts QuPath annotations (GeoJSON) into a cutting file (XML) for the Leica LMD, with the shapes assigned to wells of an 8-well strip.
Open this notebook from the `notebooks` folder of `lmd_export` (the default in Jupyter); the `qupath_to_lmd` folder next to it is found automatically.

In [ ]:
# Setup
import os
import sys
from pathlib import Path
from loguru import logger

sys.path.insert(0, str(Path.cwd().parent))   # makes the qupath_to_lmd folder importable when run from notebooks/
from qupath_to_lmd.utils import assign_samples_to_wells
from qupath_to_lmd.qc_utils import (
    load_and_QC_geojson_file,
    load_and_QC_SamplesandWells,
    create_collection,
)

logger.remove()
logger.add(sys.stdout, format="<green>{time:HH:mm:ss.SS}</green> | <level>{level}</level> | {message}")

## 1. Input

In [ ]:
# GeoJSON exported from QuPath, and the folder for all output files
geojson_path = Path("path/to/annotations.geojson")
output_folder = Path("path/to/output_folder")

# Names of the three calibration points in the GeoJSON
list_of_calibpoint_names = ["calib_1", "calib_2", "calib_3"]

# Shape names are built as <category1>_<category2>_<number>; these must match the QuPath class names
categories1 = ["Somatosensory_Cortex"]
categories2 = ["Cell"]
n_reps = 9   # number of shapes in the GeoJSON

## 2. Check the GeoJSON

In [ ]:
load_and_QC_geojson_file(geojson_path=geojson_path, list_of_calibpoint_names=list_of_calibpoint_names)

## 3. Assign shapes to wells (column-major: A1-H1, A2-H2, ...)

In [ ]:
mapping = assign_samples_to_wells(categories1, categories2, n_reps,
                                  output_folder=output_folder,
                                  filename="Samples_and_wells.txt")
print(mapping)

## 4. Check shape names and wells

In [ ]:
samples_and_wells_input = mapping
samples_and_wells = load_and_QC_SamplesandWells(
    samples_and_wells_input=samples_and_wells_input,
    geojson_path=geojson_path,
    list_of_calibpoint_names=list_of_calibpoint_names
)

## 5. Create the XML file
Written to the output folder: the XML for the LMD, an overview image (`TheCollection.png`) and a CSV of the well layout.

In [ ]:
output_folder.mkdir(parents=True, exist_ok=True)
os.chdir(output_folder)

the_collection, df_strip = create_collection(
    geojson_path=geojson_path,
    list_of_calibpoint_names=list_of_calibpoint_names,
    samples_and_wells_input=samples_and_wells_input
)